In [1]:
import numpy as np
from preprocess import preprocess_raw

# Load and preprocess EEG
edf_path = "data/raw/chb01/chb01_15.edf"
raw, raw_filtered = preprocess_raw(edf_path)

# Extract filtered signal
data = raw_filtered.get_data()   # shape: (channels, samples)
sfreq = raw_filtered.info["sfreq"]

print("Data shape:", data.shape)
print("Sampling frequency:", sfreq)

Extracting EDF parameters from data/raw/chb01/chb01_15.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 921599  =      0.000 ...  3599.996 secs...
All channel names are unique.
Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 59 - 61 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 59.35
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 59.10 Hz)
- Upper passband edge: 60.65 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 60.90 Hz)
- Filter length: 1691 samples (6.605 s)



/Users/koushik/projects/EEG-brain-functioning-and-seizures/preprocess.py:8: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_path, preload=preload)


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1691 samples (6.605 s)

Data shape: (23, 921600)
Sampling frequency: 256.0


In [2]:
def create_windows(data, sfreq, window_sec=5, step_sec=5):
    """
    Split EEG into sliding windows.

    Parameters
    ----------
    data : np.ndarray
        Shape (n_channels, n_samples)
    sfreq : float
        Sampling frequency
    window_sec : int or float
        Window length in seconds
    step_sec : int or float
        Step size in seconds

    Returns
    -------
    windows : np.ndarray
        Shape (n_windows, n_channels, window_samples)
    window_times : list of tuple
        [(start_sec, end_sec), ...]
    """
    window_samples = int(window_sec * sfreq)
    step_samples = int(step_sec * sfreq)

    n_channels, n_samples = data.shape
    windows = []
    window_times = []

    for start in range(0, n_samples - window_samples + 1, step_samples):
        end = start + window_samples
        windows.append(data[:, start:end])
        window_times.append((start / sfreq, end / sfreq))

    return np.array(windows), window_times

In [3]:
windows, window_times = create_windows(
    data=data,
    sfreq=sfreq,
    window_sec=5,
    step_sec=5
)

print("Windows shape:", windows.shape)
print("Total windows:", len(windows))
print("First window time:", window_times[0])
print("First window shape:", windows[0].shape)

Windows shape: (720, 23, 1280)
Total windows: 720
First window time: (0.0, 5.0)
First window shape: (23, 1280)


In [4]:
def label_windows(window_times, seizure_start, seizure_end):
    labels = []

    for start, end in window_times:
        has_overlap = (start < seizure_end) and (end > seizure_start)
        labels.append(1 if has_overlap else 0)

    return np.array(labels)

In [5]:
import numpy as np

seizure_start = 1732
seizure_end = 1772

labels = label_windows(window_times, seizure_start, seizure_end)

print("Labels shape:", labels.shape)
print("Seizure windows:", np.sum(labels == 1))
print("Non-seizure windows:", np.sum(labels == 0))

seizure_indices = np.where(labels == 1)[0]
normal_indices = np.where(labels == 0)[0]

print("First seizure window index:", seizure_indices[0])
print("First seizure window time:", window_times[seizure_indices[0]])
print("Last seizure window time:", window_times[seizure_indices[-1]])
print("First normal window time:", window_times[normal_indices[0]])

Labels shape: (720,)
Seizure windows: 9
Non-seizure windows: 711
First seizure window index: 346
First seizure window time: (1730.0, 1735.0)
Last seizure window time: (1770.0, 1775.0)
First normal window time: (0.0, 5.0)


In [ ]:
windows, window_times = create_windows(
    data=data,
    sfreq=sfreq,
    window_sec=5,
    step_sec=2.5   # overlap
)


In [8]:
labels = label_windows(window_times, seizure_start, seizure_end)

In [9]:
print("Total windows:", len(windows))
print("Seizure windows:", np.sum(labels == 1))
print("Non-seizure windows:", np.sum(labels == 0))

Total windows: 1439
Seizure windows: 18
Non-seizure windows: 1421
